## Import Required Libraries

In [ ]:
# Imports (Collected in one place for readability)

# Core
import numpy as np
import pandas as pd

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# Transformers (Hugging Face)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

# Scikit-learn (classical ML + metrics)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Utilities
import matplotlib.pyplot as plt
import seaborn as sns
import random

## Load the Dataset

In [ ]:
# Load the Dataset

from datasets import load_dataset

# Load the emotion dataset
dataset = load_dataset("dair-ai/emotion")

# Check dataset structure
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


## BERT-based transformers model

In [ ]:
# BERT-based transformers model

from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load pre-trained model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=6)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Tokenize the Data

In [ ]:
# Tokenize the Data

from transformers import AutoTokenizer

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Apply tokenization
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Print a sample
print(tokenized_datasets["train"][0])

{'text': 'i didnt feel humiliated', 'label': 0, 'input_ids': [101, 1045, 2134, 2102, 2514, 26608, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

## Handle Class Imbalance

In [ ]:
# Handle Class Imbalance

from collections import Counter
import torch

# Count label distribution in training data
labels = dataset["train"]["label"]
label_counts = Counter(labels)

# Compute class weights
num_samples = len(labels)
num_classes = len(label_counts)
class_weights = [num_samples / (num_classes * count) for count in label_counts.values()]
class_weights_tensor = torch.tensor(class_weights)

print(f"Class Weights: {class_weights}")

Class Weights: [0.5715102157451064, 1.2351397251814111, 2.044989775051125, 4.662004662004662, 1.3766993632765445, 0.49732686808404825]


## Define Training Arguments & Trainer

In [ ]:
# Define Training Arguments & Trainer

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",  # Evaluate after every epoch
    save_strategy="epoch",  # Save best model at the end
    learning_rate=1e-5,  # Lower learning rate improves generalization
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,  # Train for longer
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
)

## Train the model

In [ ]:
# Train the model

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.323400,0.228848
2,0.173700,0.177630
3,0.121600,0.178826
4,0.106800,0.176363
5,0.077600,0.184593


TrainOutput(global_step=5000, training_loss=0.22754581909179689, metrics={'train_runtime': 18753.6454, 'train_samples_per_second': 4.266, 'train_steps_per_second': 0.267, 'total_flos': 2.104964038656e+16, 'train_loss': 0.22754581909179689, 'epoch': 5.0})

## Evaluate the Model on Test Data

In [ ]:
# Evaluate the Model on Test Data

import evaluate

# Load accuracy metric
metric = evaluate.load("accuracy")

# Get model predictions
predictions = trainer.predict(tokenized_datasets["test"])
preds = predictions.predictions.argmax(-1)

# Compute accuracy
accuracy = metric.compute(predictions=preds, references=tokenized_datasets["test"]["label"])
print(f"Test Accuracy: {accuracy['accuracy']:.4f}")

Test Accuracy: 0.9205


## Improve Model Generalization (Fix Overfitting)

In [ ]:
# Improve Model Generalization (Fix Overfitting)

from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]  # Stop if no improvement after 1 epoch
)

## Apply Dropout Regularization in the model

In [ ]:
# Apply Dropout Regularization in the model

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=6, hidden_dropout_prob=0.3)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Analyze Model Errors (Classification Report)

In [ ]:
# Analyze Model Errors (Classification Report)

from sklearn.metrics import classification_report

labels = tokenized_datasets["test"]["label"]
print(classification_report(labels, preds, target_names=["sadness", "joy", "love", "anger", "fear", "surprise"]))

              precision    recall  f1-score   support

     sadness       0.96      0.96      0.96       581
         joy       0.95      0.95      0.95       695
        love       0.83      0.81      0.82       159
       anger       0.93      0.90      0.91       275
        fear       0.86      0.88      0.87       224
    surprise       0.70      0.77      0.73        66

    accuracy                           0.92      2000
   macro avg       0.87      0.88      0.87      2000
weighted avg       0.92      0.92      0.92      2000



## Analyze Misclassifications

In [ ]:
# Analyze Misclassifications

import pandas as pd

# Convert test data to DataFrame
df_test = dataset["test"].to_pandas()
df_test["predictions"] = preds

# Show misclassified examples
misclassified = df_test[df_test["label"] != df_test["predictions"]]
print(misclassified.sample(10))

                                                   text  label  predictions
1479  i really feel and i know the devil hates that ...      1            3
457   i cant do strappy shoes at work i just feel we...      4            5
917   i feel the need to pimp this since raini my be...      1            2
1936  im polyamorous something im starting to feel t...      2            1
863   i feel betrayed and angry and sad at the same ...      3            0
1764  i don t know how else to describe it except to...      1            2
820            i found myself feeling a bit overwhelmed      5            4
625   i am feeling overwhelmed by trying to do it al...      5            4
1087  i feel for all of you who have been supporting...      1            2
688                             i feel hated in cempaka      0            3


## Save the Model for Deployment

In [ ]:
# Save the Model for Deployment

trainer.save_model("./best_emotion_model")
tokenizer.save_pretrained("./best_emotion_model")

('./best_emotion_model/tokenizer_config.json',
 './best_emotion_model/special_tokens_map.json',
 './best_emotion_model/vocab.txt',
 './best_emotion_model/added_tokens.json',
 './best_emotion_model/tokenizer.json')

## To reload the best model later

In [ ]:
# To reload the best model later

from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("./best_emotion_model")
tokenizer = AutoTokenizer.from_pretrained("./best_emotion_model")

## Test on Custom Sentences

In [ ]:
# Test on Custom Sentences

texts = ["I'm feeling really down today.", "Wow! This is amazing!", "I am so scared to go outside."]

# Tokenize and predict
inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
outputs = model(**inputs)
predictions = outputs.logits.argmax(-1)

# Print predictions
for text, pred in zip(texts, predictions):
    print(f"Text: {text} → Predicted Emotion: {tokenized_datasets['train'].features['label'].int2str(pred.item())}")

Text: I'm feeling really down today. → Predicted Emotion: sadness
Text: Wow! This is amazing! → Predicted Emotion: surprise
Text: I am so scared to go outside. → Predicted Emotion: fear


## Extract Logits Before Applying argmax()

In [ ]:
# Extract Logits Before Applying argmax()

import torch

def predict_emotion(text):
    # Ensure the model is in evaluation mode
    model.eval()

    # Move model and input to the correct device (GPU or CPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Tokenize input text
    tokens = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)

    # Get model output
    with torch.no_grad():  # Disable gradient calculations for inference
        output = model(**tokens)  # Correct way to pass inputs to transformer models

    # Extract logits and apply argmax to get the predicted label index
    logits = output.logits  # Extract logits
    prediction = torch.argmax(logits, dim=1).item()  # Get the highest probability class

    # Emotion labels
    emotion_labels = ["sadness", "joy", "love", "anger", "fear", "surprise"]

    return emotion_labels[prediction]

# Example Predictions
print(predict_emotion("I am feeling very happy today!"))  # Expected: Joy
print(predict_emotion("I'm so scared of what will happen next."))  # Expected: Fear
print(predict_emotion("This is the worst day of my life."))  # Expected: Sadness
print(predict_emotion("Wow, this is amazing!"))  # Expected: Surprise

joy
fear
sadness
surprise


## Convert Text to TF-IDF Features

In [ ]:
# Convert Text to TF-IDF Features

from sklearn.feature_extraction.text import TfidfVectorizer

# Convert text data into TF-IDF vectors
vectorizer = TfidfVectorizer(max_features=5000)  # Limit to 5000 features for efficiency
X_train = vectorizer.fit_transform(df_train["text"])
X_test = vectorizer.transform(df_test["text"])

# Extract labels
y_train = df_train["label"]
y_test = df_test["label"]

# Check shape of transformed data
X_train.shape, X_test.shape

((16000, 5000), (2000, 5000))

## Train & Evaluate the Support Vector Machine (SVM) on Test Data

In [ ]:
# Train & Evaluate the Support Vector Machine (SVM) on Test Data

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Train SVM model
svm_model = SVC(kernel="linear")
svm_model.fit(X_train, y_train)

# Predict and evaluate
svm_preds = svm_model.predict(X_test)
svm_acc = accuracy_score(y_test, svm_preds)

print(f"SVM Accuracy: {svm_acc:.4f}")

SVM Accuracy: 0.8870


## SVM Classification Report

In [ ]:
# SVM Classification Report

print("Classification Report for SVM:")
print(classification_report(y_test, svm_preds, target_names=["sadness", "joy", "love", "anger", "fear", "surprise"]))

Classification Report for SVM:
              precision    recall  f1-score   support

     sadness       0.93      0.92      0.93       581
         joy       0.88      0.95      0.91       695
        love       0.82      0.69      0.75       159
       anger       0.89      0.88      0.88       275
        fear       0.85      0.86      0.86       224
    surprise       0.74      0.56      0.64        66

    accuracy                           0.89      2000
   macro avg       0.85      0.81      0.83      2000
weighted avg       0.89      0.89      0.88      2000



## Analyze Misclassifications for SVM

In [ ]:
# Analyze Misclassifications for SVM

import pandas as pd

# Convert test data to DataFrame
df_test_svm = dataset["test"].to_pandas()

# Add SVM Predictions to the DataFrame
df_test_svm["svm_predictions"] = svm_model.predict(X_test_tfidf)

# Show misclassified examples
misclassified_svm = df_test_svm[df_test_svm["label"] != df_test_svm["svm_predictions"]]

# Display random 10 misclassified examples
print("Misclassified Examples (SVM Model):")
print(misclassified_svm.sample(10))

Misclassified Examples (SVM Model):
                                                   text  label  \
103   i feel agitated with myself that i did not for...      4   
1048  i wonder if the homeowners would feel weird if...      5   
501   when we rearranged furniture in our flat and g...      3   
1592        i have strong feelings about being faithful      2   
575   i feel not having a generous spirit or a forgi...      2   
625   i am feeling overwhelmed by trying to do it al...      5   
1928  i feel inside cause life is like a game someti...      4   
1296         i that it feels like she is being tortured      4   
869   i feel like if people accepted that wed get al...      2   
823   i dont remember how january was like last year...      3   

      svm_predictions  
103                 3  
1048                4  
501                 1  
1592                1  
575                 1  
625                 4  
1928                0  
1296                3  
869                 1

## Test on Custom Sentences for SVM

In [ ]:
# Test on Custom Sentences for SVM

# Define custom sentences
custom_texts = [
    "I am feeling very happy today!",
    "This is the worst day of my life.",
    "I can't stop smiling, this is the best surprise ever!",
    "I am so scared to go outside alone.",
    "I feel so loved and appreciated today.",
    "Why do you always make me so angry?",
    "I feel like crying all day long."
]

# Transform custom sentences using the same TF-IDF vectorizer
custom_tfidf_features = tfidf_vectorizer.transform(custom_texts)

# Predict using SVM
svm_custom_preds = svm_model.predict(custom_tfidf_features)

# Emotion Labels Mapping
emotion_labels = ["sadness", "joy", "love", "anger", "fear", "surprise"]

# Convert predictions to emotion labels
svm_custom_preds_labels = [emotion_labels[pred] for pred in svm_custom_preds]

# Print Predictions
print("Custom Sentence Predictions (SVM):\n")
for text, pred_label in zip(custom_texts, svm_custom_preds_labels):
    print(f"Sentence: {text}")
    print(f"SVM Prediction: {pred_label}\n")

Custom Sentence Predictions (SVM):

Sentence: I am feeling very happy today!
SVM Prediction: joy

Sentence: This is the worst day of my life.
SVM Prediction: joy

Sentence: I can't stop smiling, this is the best surprise ever!
SVM Prediction: joy

Sentence: I am so scared to go outside alone.
SVM Prediction: fear

Sentence: I feel so loved and appreciated today.
SVM Prediction: love

Sentence: Why do you always make me so angry?
SVM Prediction: anger

Sentence: I feel like crying all day long.
SVM Prediction: joy



## Extract Logits Before Applying argmax() to SVM

In [ ]:
# Extract Logits Before Applying argmax() to SVM

import numpy as np

def predict_emotion_svm(text):
    """
    Function to predict the emotion using the trained SVM model.
    Extracts logits (decision function values) before applying argmax().
    """
    # Transform input text using the same TF-IDF vectorizer
    text_tfidf = tfidf_vectorizer.transform([text])

    # Get model output (decision function for probabilities)
    logits = svm_model.decision_function(text_tfidf)

    # Apply argmax to get the predicted label index
    prediction = np.argmax(logits)

    # Emotion labels
    emotion_labels = ["sadness", "joy", "love", "anger", "fear", "surprise"]

    return emotion_labels[prediction], logits  # Returning logits along with prediction

# Example Predictions with Logits for SVM
custom_sentences = [
    "I am feeling very happy today!",  # Expected: Joy
    "I'm so scared of what will happen next.",  # Expected: Fear
    "This is the worst day of my life.",  # Expected: Sadness
    "Wow, this is amazing!"  # Expected: Surprise
]

# Make predictions and extract logits
print("SVM Predictions with Logits:\n")
for text in custom_sentences:
    prediction, logits = predict_emotion_svm(text)
    print(f"Sentence: {text}")
    print(f"SVM Prediction: {prediction}")
    print(f"Logits: {logits}\n")  # Raw model decision values

SVM Predictions with Logits:

Sentence: I am feeling very happy today!
SVM Prediction: joy
Logits: [[ 4.21734192  5.29313949  0.75707932  2.84660651  1.83615835 -0.27445076]]

Sentence: I'm so scared of what will happen next.
SVM Prediction: fear
Logits: [[ 4.13534549  2.99247187  0.7420411   1.8344909   5.29496232 -0.26584384]]

Sentence: This is the worst day of my life.
SVM Prediction: joy
Logits: [[ 4.22671562  5.25770218  0.76205444  3.21362757  1.8341     -0.2643637 ]]

Sentence: Wow, this is amazing!
SVM Prediction: joy
Logits: [[ 2.80986026  5.28427686 -0.27432975  1.75275479  0.74291711  4.28844537]]



## Train & Evaluate the Random Forest on Test Data

In [ ]:
# Train & Evaluate the Random Forest on Test Data

from sklearn.ensemble import RandomForestClassifier

# Train Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict and evaluate
rf_preds = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)

print(f"Random Forest Accuracy: {rf_acc:.4f}")

Random Forest Accuracy: 0.8725


## Random Forest Classification Report

In [ ]:
# Random Forest Classification Report

print("Classification Report for Random Forest:")
print(classification_report(y_test, rf_preds, target_names=["sadness", "joy", "love", "anger", "fear", "surprise"]))

Random Forest Accuracy: 0.8800


## Analyze Misclassifications for Random Forest

In [ ]:
# Analyze Misclassifications for Random Forest

import pandas as pd

# Convert test data to DataFrame
df_test_rf = dataset["test"].to_pandas()

# Add Random Forest Predictions to the DataFrame
df_test_rf["rf_predictions"] = rf_model.predict(X_test_tfidf)

# Show misclassified examples
misclassified_rf = df_test_rf[df_test_rf["label"] != df_test_rf["rf_predictions"]]

# Display random 10 misclassified examples
print("Misclassified Examples (Random Forest Model):")
print(misclassified_rf.sample(10))

Misclassified Examples (Random Forest Model):
                                                   text  label  rf_predictions
660   i was playing a sport in an advanced pe class ...      3               1
1530  i feel furious at love because i really though...      3               1
1146  i feel affirmed gracious sensuous and will hav...      2               1
119   i feel like i know who most of them are by now...      1               0
1402                     i just keep on feeling blessed      2               1
565   i feel like an ugly monster where i cannot sho...      0               4
1791  i did a body scan and realized that everything...      5               1
941   i still feel confused and guilty about the who...      4               0
466              i feel his hand on me to stay faithful      2               1
468                        i cant help feeling this way      0               1


## Test on Custom Sentences for Random Forest

In [ ]:
# Test on Custom Sentences for Random Forest

# Define custom sentences
custom_texts = [
    "I am feeling very happy today!",
    "This is the worst day of my life.",
    "I can't stop smiling, this is the best surprise ever!",
    "I am so scared to go outside alone.",
    "I feel so loved and appreciated today.",
    "Why do you always make me so angry?",
    "I feel like crying all day long."
]

# Transform custom sentences using the same TF-IDF vectorizer
custom_tfidf_features = tfidf_vectorizer.transform(custom_texts)

# Predict using Random Forest
rf_custom_preds = rf_model.predict(custom_tfidf_features)

# Emotion Labels Mapping
emotion_labels = ["sadness", "joy", "love", "anger", "fear", "surprise"]

# Convert predictions to emotion labels
rf_custom_preds_labels = [emotion_labels[pred] for pred in rf_custom_preds]

# Print Predictions
print("Custom Sentence Predictions (Random Forest):\n")
for text, pred_label in zip(custom_texts, rf_custom_preds_labels):
    print(f"Sentence: {text}")
    print(f"Random Forest Prediction: {pred_label}\n")

Custom Sentence Predictions (Random Forest):

Sentence: I am feeling very happy today!
Random Forest Prediction: joy

Sentence: This is the worst day of my life.
Random Forest Prediction: joy

Sentence: I can't stop smiling, this is the best surprise ever!
Random Forest Prediction: joy

Sentence: I am so scared to go outside alone.
Random Forest Prediction: fear

Sentence: I feel so loved and appreciated today.
Random Forest Prediction: love

Sentence: Why do you always make me so angry?
Random Forest Prediction: anger

Sentence: I feel like crying all day long.
Random Forest Prediction: joy



## Extract Logits Before Applying argmax() to Random Forest

In [ ]:
# Extract Logits Before Applying argmax() to Random Forest

import numpy as np

def predict_emotion_rf(text):
    """
    Function to predict the emotion using the trained Random Forest model.
    Extracts logits (probabilities) before applying argmax().
    """
    # Transform input text using the same TF-IDF vectorizer
    text_tfidf = tfidf_vectorizer.transform([text])

    # Get model output (predict_proba gives probability distribution over classes)
    logits = rf_model.predict_proba(text_tfidf)[0]  # Extracting probabilities

    # Apply argmax to get the predicted label index
    prediction = np.argmax(logits)

    # Emotion labels
    emotion_labels = ["sadness", "joy", "love", "anger", "fear", "surprise"]

    return emotion_labels[prediction], logits  # Returning logits along with prediction

# Example Predictions with Logits for Random Forest
custom_sentences = [
    "I am feeling very happy today!",  # Expected: Joy
    "I'm so scared of what will happen next.",  # Expected: Fear
    "This is the worst day of my life.",  # Expected: Sadness
    "Wow, this is amazing!"  # Expected: Surprise
]

# Make predictions and extract logits
print("Random Forest Predictions with Logits:\n")
for text in custom_sentences:
    prediction, logits = predict_emotion_rf(text)
    print(f"Sentence: {text}")
    print(f"RF Prediction: {prediction}")
    print(f"Logits (Probabilities): {logits}\n")  # Raw model probability values

Random Forest Predictions with Logits:

Sentence: I am feeling very happy today!
RF Prediction: joy
Logits (Probabilities): [0.1  0.76 0.05 0.06 0.03 0.  ]

Sentence: I'm so scared of what will happen next.
RF Prediction: fear
Logits (Probabilities): [0.13 0.05 0.01 0.02 0.79 0.  ]

Sentence: This is the worst day of my life.
RF Prediction: joy
Logits (Probabilities): [0.2  0.53 0.03 0.1  0.13 0.01]

Sentence: Wow, this is amazing!
RF Prediction: joy
Logits (Probabilities): [0.04 0.6  0.   0.02 0.   0.34]



## Train & Evaluate the Logistic Regression Model on Test Data

In [ ]:
# Train & Evaluate the Logistic Regression Model on Test Data

# Initialize Logistic Regression model
log_reg_model = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42)

# Train the model on TF-IDF features
log_reg_model.fit(X_train_tfidf, y_train)

# Predict on Test Data
log_reg_preds = log_reg_model.predict(X_test_tfidf)

# Evaluate the model
log_reg_accuracy = accuracy_score(y_test, log_reg_preds)
print(f"✅ Logistic Regression Test Accuracy: {log_reg_accuracy:.4f}")

✅ Logistic Regression Test Accuracy: 0.8690


## Classification Report for Logistic Regression

In [ ]:
# Classification Report for Logistic Regression

print("Classification Report for Logistic Regression:")
print(classification_report(y_test, lr_preds, target_names=["sadness", "joy", "love", "anger", "fear", "surprise"]))

Classification Report for Logistic Regression:
              precision    recall  f1-score   support

     sadness       0.90      0.93      0.91       581
         joy       0.85      0.96      0.90       695
        love       0.82      0.62      0.71       159
       anger       0.89      0.83      0.86       275
        fear       0.88      0.79      0.83       224
    surprise       0.80      0.53      0.64        66

    accuracy                           0.87      2000
   macro avg       0.86      0.78      0.81      2000
weighted avg       0.87      0.87      0.87      2000



## Analyze Misclassifications for Logistic Regression Model

In [ ]:
# Analyze Misclassifications for Logistic Regression Model

import pandas as pd

# Convert test dataset to Pandas DataFrame
df_test_log_reg = dataset["test"].to_pandas()

# Add Logistic Regression Predictions to the DataFrame
df_test_log_reg["log_reg_predictions"] = log_reg_model.predict(X_test_tfidf)

# Show misclassified examples
misclassified_log_reg = df_test_log_reg[df_test_log_reg["label"] != df_test_log_reg["log_reg_predictions"]]

# Display 10 randomly selected misclassified examples
print("Misclassified Examples (Logistic Regression Model):")
print(misclassified_log_reg.sample(10))

Misclassified Examples (Logistic Regression Model):
                                                   text  label  \
433   i know that i have it nowhere near as worse as...      4   
74    i were to go overseas or cross the border then...      2   
1936  im polyamorous something im starting to feel t...      2   
222   i think i wanted audiences to feel impressed i...      5   
96    i love neglecting this blog but sometimes i fe...      2   
715   i get to be creative if i feel like it or just...      2   
242   i see you on the pitchers mound at our little ...      4   
193   i really dont like quinn because i feel like s...      3   
1387  im not sure but theres nothing that will get a...      2   
121                        made a wonderfull new friend      1   

      log_reg_predictions  
433                     0  
74                      1  
1936                    1  
222                     1  
96                      1  
715                     1  
242                     0

## Test on Custom Sentences for Logistic Regression

In [ ]:
# Test on Custom Sentences for Logistic Regression

# Define custom sentences
custom_texts = [
    "I am feeling very happy today!",
    "This is the worst day of my life.",
    "I can't stop smiling, this is the best surprise ever!",
    "I am so scared to go outside alone.",
    "I feel so loved and appreciated today.",
    "Why do you always make me so angry?",
    "I feel like crying all day long."
]

# Transform custom sentences using the same TF-IDF vectorizer
custom_tfidf_features = tfidf_vectorizer.transform(custom_texts)

# Predict using Logistic Regression
log_reg_custom_preds = log_reg_model.predict(custom_tfidf_features)

# Emotion Labels Mapping
emotion_labels = ["sadness", "joy", "love", "anger", "fear", "surprise"]

# Convert predictions to emotion labels
log_reg_custom_preds_labels = [emotion_labels[pred] for pred in log_reg_custom_preds]

# Print Predictions
print("Custom Sentence Predictions (Logistic Regression):\n")
for text, pred_label in zip(custom_texts, log_reg_custom_preds_labels):
    print(f"Sentence: {text}")
    print(f"Logistic Regression Prediction: {pred_label}\n")

Custom Sentence Predictions (Logistic Regression):

Sentence: I am feeling very happy today!
Logistic Regression Prediction: joy

Sentence: This is the worst day of my life.
Logistic Regression Prediction: joy

Sentence: I can't stop smiling, this is the best surprise ever!
Logistic Regression Prediction: joy

Sentence: I am so scared to go outside alone.
Logistic Regression Prediction: fear

Sentence: I feel so loved and appreciated today.
Logistic Regression Prediction: love

Sentence: Why do you always make me so angry?
Logistic Regression Prediction: anger

Sentence: I feel like crying all day long.
Logistic Regression Prediction: sadness



## Extract Logits Before Applying argmax() to Logistic Regression

In [ ]:
# Extract Logits Before Applying argmax() to Logistic Regression

import numpy as np

def predict_emotion_log_reg(text):
    """
    Function to predict the emotion using the trained Logistic Regression model.
    Extracts logits (probabilities) before applying argmax().
    """
    # Transform input text using the same TF-IDF vectorizer
    text_tfidf = tfidf_vectorizer.transform([text])

    # Get model output (predict_proba gives probability distribution over classes)
    logits = log_reg_model.predict_proba(text_tfidf)[0]  # Extracting probabilities

    # Apply argmax to get the predicted label index
    prediction = np.argmax(logits)

    # Emotion labels
    emotion_labels = ["sadness", "joy", "love", "anger", "fear", "surprise"]

    return emotion_labels[prediction], logits  # Returning logits along with prediction

# Example Predictions with Logits for Logistic Regression
custom_sentences = [
    "I am feeling very happy today!",  # Expected: Joy
    "I'm so scared of what will happen next.",  # Expected: Fear
    "This is the worst day of my life.",  # Expected: Sadness
    "Wow, this is amazing!"  # Expected: Surprise
]

# Make predictions and extract logits
print("Logistic Regression Predictions with Logits:\n")
for text in custom_sentences:
    prediction, logits = predict_emotion_log_reg(text)
    print(f"Sentence: {text}")
    print(f"Logistic Regression Prediction: {prediction}")
    print(f"Logits (Probabilities): {logits}\n")  # Raw model probability values

Logistic Regression Predictions with Logits:

Sentence: I am feeling very happy today!
Logistic Regression Prediction: joy
Logits (Probabilities): [0.07556277 0.85000386 0.01456523 0.03092256 0.02330232 0.00564325]

Sentence: I'm so scared of what will happen next.
Logistic Regression Prediction: fear
Logits (Probabilities): [0.0754142  0.05694343 0.02117282 0.03144245 0.8017663  0.0132608 ]

Sentence: This is the worst day of my life.
Logistic Regression Prediction: joy
Logits (Probabilities): [0.28885688 0.39450347 0.06980073 0.14504974 0.07739482 0.02439436]

Sentence: Wow, this is amazing!
Logistic Regression Prediction: joy
Logits (Probabilities): [0.05419698 0.51559954 0.03195987 0.04481148 0.03571615 0.31771598]



## We can now combine train & evaluate the Bert, SVM, Random Fore...g a simple majority voting for an Ensemble Approach on Test Data

In [ ]:
# We can now combine train & evaluate the Bert, SVM, Random Forest, and Logistic Regression using a simple majority voting for an Ensemble Approach on Test Data

# Stack predictions from all models
predictions_stack = np.column_stack((svm_preds, rf_preds, log_reg_preds, bert_preds))

# Majority voting: select the most common prediction for each instance
ensemble_preds, _ = mode(predictions_stack, axis=1)
ensemble_preds = ensemble_preds.flatten()  # Convert to 1D array

# Evaluate ensemble accuracy
ensemble_acc = accuracy_score(y_test, ensemble_preds)

print(f"\nEnsemble Model Accuracy (BERT + SVM + RF + LR): {ensemble_acc:.4f}")


Ensemble Model Accuracy (BERT + SVM + RF + LR): 0.8980


## Ensemble Model Classification Report (Bert, SVM, Random Forest, and Logistic Regression)

In [ ]:
# Ensemble Model Classification Report (Bert, SVM, Random Forest, and Logistic Regression)

print("\nEnsemble Model Classification Report:")
print(classification_report(y_test, ensemble_preds, target_names=["sadness", "joy", "love", "anger", "fear", "surprise"]))


Ensemble Model Classification Report:
              precision    recall  f1-score   support

     sadness       0.93      0.96      0.94       581
         joy       0.88      0.97      0.92       695
        love       0.84      0.64      0.73       159
       anger       0.93      0.87      0.90       275
        fear       0.87      0.86      0.87       224
    surprise       0.87      0.50      0.63        66

    accuracy                           0.90      2000
   macro avg       0.89      0.80      0.83      2000
weighted avg       0.90      0.90      0.89      2000



## Analyze Misclassifications for Ensemble Model (Bert, SVM, Random Forest, and Logistic Regression)

In [ ]:
# Analyze Misclassifications for Ensemble Model (Bert, SVM, Random Forest, and Logistic Regression)

# Convert test dataset to Pandas DataFrame
df_test_ensemble = dataset["test"].to_pandas()

# Get predictions from individual models
svm_preds = svm_model.predict(X_test_tfidf)
rf_preds = rf_model.predict(X_test_tfidf)
log_reg_preds = log_reg_model.predict(X_test_tfidf)

# Get predictions from the BERT model
bert_preds = trainer.predict(tokenized_datasets["test"]).predictions.argmax(-1)

# Perform Majority Voting for Ensemble Prediction
ensemble_preds = np.array([svm_preds, rf_preds, log_reg_preds, bert_preds])  # Stack predictions
final_ensemble_preds = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=ensemble_preds)

# Add Ensemble Predictions to the DataFrame
df_test_ensemble["ensemble_predictions"] = final_ensemble_preds

# Show misclassified examples
misclassified_ensemble = df_test_ensemble[df_test_ensemble["label"] != df_test_ensemble["ensemble_predictions"]]

# Display 10 randomly selected misclassified examples
print("Misclassified Examples (Ensemble Model - BERT + SVM + RF + Logistic Regression):")
print(misclassified_ensemble.sample(10))

Misclassified Examples (Ensemble Model - BERT + SVM + RF + Logistic Regression):
                                                   text  label  \
693   i can say is that as long as you enjoy the sto...      0   
1533  i actually was in a meeting last week where so...      3   
1714           i also do feel passionate about teaching      2   
206   i wish to know whether i should feel sympathet...      2   
476   i feel quite helpless in all of this so prayer...      0   
254   i feel blessed beyond blessed to share my life...      2   
1467  i seek out pain to feel tortured just to feel ...      4   
433   i know that i have it nowhere near as worse as...      4   
828      i feel unprotected even while travelling alone      4   
861   i feel assaulted by this shit storm of confusi...      4   

      ensemble_predictions  
693                      3  
1533                     0  
1714                     1  
206                      1  
476                      4  
254               

## Test on Custom Sentences for Ensemble Model (Bert, SVM, Random Forest, and Logistic Regression)

In [ ]:
# Test on Custom Sentences for Ensemble Model (Bert, SVM, Random Forest, and Logistic Regression)

# Define custom sentences
custom_texts = [
    "I am feeling very happy today!",
    "This is the worst day of my life.",
    "I can't stop smiling, this is the best surprise ever!",
    "I am so scared to go outside alone.",
    "I feel so loved and appreciated today.",
    "Why do you always make me so angry?",
    "I feel like crying all day long."
]

# Transform custom sentences using the same TF-IDF vectorizer for ML models
custom_tfidf_features = tfidf_vectorizer.transform(custom_texts)

# Get predictions from ML models
svm_custom_preds = svm_model.predict(custom_tfidf_features)
rf_custom_preds = rf_model.predict(custom_tfidf_features)
log_reg_custom_preds = log_reg_model.predict(custom_tfidf_features)

# Get predictions from the BERT model
custom_tokenized_inputs = tokenizer(custom_texts, return_tensors="pt", padding=True, truncation=True)
custom_tokenized_inputs = {k: v.to(model.device) for k, v in custom_tokenized_inputs.items()}  # Move to device

with torch.no_grad():
    bert_logits = model(**custom_tokenized_inputs).logits
    bert_custom_preds = bert_logits.argmax(dim=1).cpu().numpy()

# Perform Majority Voting for Ensemble Prediction
ensemble_preds = np.array([svm_custom_preds, rf_custom_preds, log_reg_custom_preds, bert_custom_preds])  # Stack predictions
final_ensemble_preds = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=ensemble_preds)

# Emotion Labels Mapping
emotion_labels = ["sadness", "joy", "love", "anger", "fear", "surprise"]

# Convert predictions to emotion labels
ensemble_custom_preds_labels = [emotion_labels[pred] for pred in final_ensemble_preds]

# Print Predictions
print("Custom Sentence Predictions (Ensemble Model - BERT + SVM + RF + Logistic Regression):\n")
for text, pred_label in zip(custom_texts, ensemble_custom_preds_labels):
    print(f"Sentence: {text}")
    print(f"Ensemble Model Prediction: {pred_label}\n")

Custom Sentence Predictions (Ensemble Model - BERT + SVM + RF + Logistic Regression):

Sentence: I am feeling very happy today!
Ensemble Model Prediction: joy

Sentence: This is the worst day of my life.
Ensemble Model Prediction: joy

Sentence: I can't stop smiling, this is the best surprise ever!
Ensemble Model Prediction: joy

Sentence: I am so scared to go outside alone.
Ensemble Model Prediction: fear

Sentence: I feel so loved and appreciated today.
Ensemble Model Prediction: love

Sentence: Why do you always make me so angry?
Ensemble Model Prediction: anger

Sentence: I feel like crying all day long.
Ensemble Model Prediction: sadness



## Extract Logits Before Applying argmax() to Ensemble Model (Bert, SVM + Random Forest + Logistic Regression)

In [ ]:
# Extract Logits Before Applying argmax() to Ensemble Model (Bert, SVM + Random Forest + Logistic Regression)

def predict_emotion_ensemble(text):
    """
    Function to predict the emotion using the Ensemble Model (BERT + SVM + RF + Logistic Regression).
    Extracts logits (probabilities/decision function values) before applying argmax().
    """
    # Transform input text using the same TF-IDF vectorizer for ML models
    text_tfidf = tfidf_vectorizer.transform([text])

    # Get model outputs (probabilities/logits)
    svm_logits = svm_model.decision_function(text_tfidf)  # Decision function values for SVM
    rf_logits = rf_model.predict_proba(text_tfidf)[0]  # Probabilities for Random Forest
    log_reg_logits = log_reg_model.predict_proba(text_tfidf)[0]  # Probabilities for Logistic Regression

    # Get logits from BERT model
    tokenized_text = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    tokenized_text = {k: v.to(model.device) for k, v in tokenized_text.items()}  # Move to device

    with torch.no_grad():
        bert_logits = model(**tokenized_text).logits.cpu().numpy()[0]  # Extract logits from BERT

    # Normalize SVM logits to match probability scale
    svm_probs = np.exp(svm_logits) / np.sum(np.exp(svm_logits), axis=1, keepdims=True)

    # Average the logits for final ensemble decision
    ensemble_logits = (svm_probs[0] + rf_logits + log_reg_logits + bert_logits) / 4  # Averaging

    # Apply argmax to get the predicted label index
    prediction = np.argmax(ensemble_logits)

    # Emotion labels
    emotion_labels = ["sadness", "joy", "love", "anger", "fear", "surprise"]

    return emotion_labels[prediction], ensemble_logits  # Returning logits along with prediction

# Example Predictions with Logits for Ensemble Model
custom_sentences = [
    "I am feeling very happy today!",  # Expected: Joy
    "I'm so scared of what will happen next.",  # Expected: Fear
    "This is the worst day of my life.",  # Expected: Sadness
    "Wow, this is amazing!"  # Expected: Surprise
]

# Make predictions and extract logits
print("Ensemble Model Predictions with Logits (BERT + SVM + RF + Logistic Regression):\n")
for text in custom_sentences:
    prediction, logits = predict_emotion_ensemble(text)
    print(f"Sentence: {text}")
    print(f"Ensemble Model Prediction: {prediction}")
    print(f"Logits (Averaged Probabilities): {logits}\n")  # Raw model decision values

Ensemble Model Predictions with Logits (BERT + SVM + RF + Logistic Regression):

Sentence: I am feeling very happy today!
Ensemble Model Prediction: joy
Logits (Averaged Probabilities): [-0.2570371   2.38853983 -0.20290386 -0.38366643 -0.37555413 -0.28720264]

Sentence: I'm so scared of what will happen next.
Ensemble Model Prediction: fear
Logits (Averaged Probabilities): [-0.37362869 -0.38495599 -0.35005543 -0.26266308  2.23493997 -0.2599686 ]

Sentence: This is the worst day of my life.
Ensemble Model Prediction: sadness
Logits (Averaged Probabilities): [ 1.6625196  -0.12750439 -0.57734552  0.55725768 -0.1043694  -0.63198028]

Sentence: Wow, this is amazing!
Ensemble Model Prediction: surprise
Logits (Averaged Probabilities): [-0.38694778  0.86072292 -0.3691757  -0.37401361 -0.01579849  1.4125763 ]

